In [5]:
import sys
import numpy as np
import re
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import warnings

warnings.filterwarnings('ignore')

In [68]:
def clean_text(text):
    text = re.sub(r'&[^ ]*?;', ' ', text)
    text = re.sub(r'[^A-Za-z0-9\s]', ' ', text)
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def preprocess(text):
    cities, sections, headings, categories = [], [], [], []
    for line in text:
        if line.strip():
            rows = json.loads(line)
            cities.append(rows.get('city', '').replace('.en','').strip())
            sections.append(rows.get('section', '').strip())
            #headings.append(rows.get('heading', '').lower())
            headings.append(clean_text(rows.get('heading', '').lower()))
            categories.append(rows.get('category', '').strip())
    
    map_idx_list = [idx for idx,value in enumerate(headings) if value=='map']
    headings_1 = [headings[i] for i in range(len(headings)) if i not in map_idx_list]
    cities_1 = [cities[i] for i in range(len(cities)) if i not in map_idx_list]
    sections_1 = [sections[i] for i in range(len(sections)) if i not in map_idx_list]
    categories_1 = [categories[i] for i in range(len(categories)) if i not in map_idx_list]

    combined_text = [f'{s} {c} {h}' for c,s,h in zip(cities_1,sections_1,headings_1)]
    if all(x=='' for x in categories_1):
        return combined_text
    else:
        return combined_text, categories_1

In [69]:

def train():
    with open('training.json', 'r') as f:
        data = f.readlines()
    data = data[1:]
    combined_text, categories = preprocess(data)
    label_dict = {value:key for key,value in enumerate(set(categories))}
    labels = [label_dict[category] for category in categories]
    lr_svc_model = Pipeline([('vec', TfidfVectorizer(min_df=5,max_df=0.90)), ('svc', LinearSVC())])
    X_train,X_test,y_train,y_test = train_test_split(combined_text, labels, test_size=0.1)
    lr_svc_model.fit(X_train, y_train)
    lr_svc_pred = lr_svc_model.predict(X_test)
    return f1_score(y_test, lr_svc_pred, average="weighted")
    #return lr_svc_model, label_dict


In [70]:
if __name__ == "__main__":
    #in_data = sys.stdin.read().strip().split('\n')
    #combined_text = preprocess(in_data[1:])
    scores = []
    for i in range(10):
        scores.append(train())
    print(f'Avg score: {sum(scores)/len(scores)}')
    #nb_model, label_dict = train()
    #pred = nb_model.predict(combined_text)
    #label_dict = {key:value for value,key in label_dict.items()}

    #for cat in pred:
    #    print(label_dict[cat])

Avg score: 0.874113273088474


Avg score - no clean text: 0.8339688722743189  
Avg score - with clean text: 0.8353221791884047  
Avg score - with numbers included: 0.8362060044914198

In [ ]:
from collections import Counter

def preprocess_1(text):
    cities, sections, headings, categories = [], [], [], []
    for line in text:
        if line.strip():
            rows = json.loads(line)
            cities.append(rows.get('city', '').replace('.en','').strip())
            sections.append(rows.get('section', '').strip())
            headings.append(clean_text(rows.get('heading', '').lower()))
            categories.append(rows.get('category', '').strip())
    return headings, cities, sections, categories
    
with open('training.json', 'r') as f:
    data = f.readlines()
data = data[1:]
headings, cities, sections, categories = preprocess_1(data)

map_idx_list = [idx for idx,value in enumerate(headings) if value=='map']
map_cities = [cities[i] for i in map_idx_list]
map_sections = [sections[i] for i in map_idx_list]
map_categories = [categories[i] for i in map_idx_list]

#print(Counter(map_cities))
#print(Counter(map_sections))
#print(Counter(map_categories))

2650
